# Chapter 2 - Lab 3: Portfolio Rebalancing Agent with Episodic Memory

In this tutorial, we'll use an agent responsible for generating clear and explainable portfolio rebalancing recommendations.

First, we will build a discussion with the agent: **Conversation History**. This will constitute our **working memory**. This section is based on the implementation introduced in Chapter 2 – Lab 2. We include it here to construct our episodic memory.

Next, we will use a specific prompt to extract the most important information from this conversation history to build our **Episodic Memory**:
* A summary of the conversation
* A trajectory of the ptf rebalancing allocations depending on the various market situations
* What worked
* What didn't work
* Key insights

We will store then all this information in a vector database ==> **ChromaDB**

In the following conversations, we will extract from the vector DB the information that is most relevant to the user's query.

The retrieved items correspond to past episodes that are inserted into the prompt before issuing the new request to the agent. This allows the agent to evaluate the query not only based on its own capabilities, but also by leveraging relevant past experiences.


**Key components of long-term memory:**

* Episodic Memory = Specific experiences from the past

* Semantic memory = Factual knowledge

* Procedural memory = "How to" for example "How to perform a KYC check"

# Install libs

In [1]:
!pip install openai-agents -q

In [2]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [3]:
from agents import Agent, Runner, function_tool, SQLiteSession
import nest_asyncio
nest_asyncio.apply()

# Rebalancing Portfolio Agent

Create portfolio rebalancing agent.

In this agent, I didn't add tools to interact with or any other reflective mechanisms. It's a simple agent that I'll be using to build a memory through the conversation.


In [4]:
portfolio_agent = Agent(
    name="Portfolio Rebalancing Agent",
    instructions="""You are a Portfolio Rebalancing Agent for a professional investor.
Your goal is to propose clear, explainable rebalancing actions that keep the portfolio aligned with the target risk/return profile and constraints.

Your role
- Analyze the current portfolio (weights, asset classes, sectors, factors, cash).
- Incorporate market regime information (volatility, trends, macro environment).
- Compare current allocation to:
    - Target allocation or risk budget (if provided),
    - Or to a reasonable diversified allocation based on the information you have.
- Propose specific rebalancing trades with approximate percentage changes and short rationales.
- Prioritize diversification, risk control, and alignment with the investor’s objectives.
- Keep answers concise and structured, and state assumptions when information is missing.
- Treat the allocation provided in my prompot as the portfolio's current allocation.

Always explain your reasoning.""",
    model="gpt-4o-mini"
)


You can of course imagine an agent using tools connecting to:
* Your database (or other sources), to extract the current situation of your portfolio.
* Market data to compute volatility, trend....

This way, your agent can analyze your current portfolio allocation and recommend appropriate rebalancing actions using real market data.

# Set up Working Memory

Let’s instantiate the working memory, referred to as a session in the OpenAI Agent SDK:

In [5]:
# Create a session instance that will persist across runs
session_id = "conversation_ptf"
session_ptf_rebal = SQLiteSession(session_id)

The SQLiteSession class provides a session mechanism that stores the interaction history in a SQLite database, enabling persistence between runs. The variable session_id uniquely names the session and allows the system to associate stored messages and state with this specific conversation.

## User Query 1

We'll add the session memory parameter to the agent's run. This ensures that the entire interaction is stored and maintained as part of the working memory.

We provide the agent with the current market context, such as the volatility regime and market trend, along with the current portfolio allocation, and ask it to recommend appropriate rebalancing actions.


In [6]:
input_user_1 = """Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?"""

output_1 = await Runner.run(
    starting_agent=portfolio_agent,
    input=input_user_1,
    session=session_ptf_rebal
)

print(f"Agent Decision:\n{output_1.final_output}\n")

Agent Decision:
### Current Portfolio Analysis:
- **Stocks**: 70%
- **Bonds**: 20%
- **Cash**: 10%

### Target Allocation: 
Given the high volatility and declining market trend, a more conservative allocation is advisable, prioritizing capital preservation and risk control.

### Recommended Adjusted Allocation:
- **Stocks**: 50%
- **Bonds**: 40%
- **Cash**: 10%

### Rationale:
1. **High Volatility**: The current high volatility regime suggests a need to reduce exposure to riskier assets like stocks.
2. **Declining Market Trend**: A downtrend in the overall market favors a more defensive strategy (increasing bond exposure).

### Proposed Rebalancing Trades:
1. **Sell Stocks**: Decrease exposure from 70% to 50%
   - **Action**: Sell 20% of equity.
   - **Rationale**: Reducing stock exposure will lower risk, as stocks are more volatile during market declines.

2. **Buy Bonds**: Increase exposure from 20% to 40%
   - **Action**: Allocate the 20% from stocks into bonds.
   - **Rationale**: 

The agent begins by outlining the current portfolio allocation, then states its assumptions and presents a recommended target allocation by asset class. It also provides a detailed rationale for each proposed adjustment:
* Reduce stock allocation from 70% to 50%
* Increase bond allocation from 20% to 40%
* Maintain cash at 10%


### Display Working Memory

Here we display the working memory at this point in the interaction with the agent:

In [7]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Stocks**: 70%
- **Bonds**: 20%
- **Cash**: 10%

### Target Allocation: 
Given the high volatility and declining market trend, a more conservative allocation is advisable, prioritizing capital preservation and risk control.

### Recommended Adjusted Allocation:
- **Stocks**: 50%
- **Bonds**: 40%
- **Cash**: 10%

### Rationale:
1. **High Volatility**: The current high volatility regime suggests a need to reduce exposure to riskier assets like stocks.
2. **Declining Market Trend**: A downtrend in the overall market favors a more defensive strategy (increasing bond exposure).

### Proposed Rebalancing Trades:
1. **Sell Stocks**: 

## User Query 2

We now ask a follow-up question while keeping the session parameter during execution, ensuring that the conversation history is preserved:

In [8]:
input_user_2="In this case, I'm reducing equity to 50%, increasing bonds to 35% and cash to 15%."

output_2 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_2,
        session=session_ptf_rebal
    )

print(f"{output_2.final_output}\n")

### Revised Portfolio Analysis:
- **Current Portfolio**: 
  - Stocks: 70% 
  - Bonds: 20% 
  - Cash: 10%
  
- **Proposed Adjusted Allocation**:
  - Stocks: 50%
  - Bonds: 35%
  - Cash: 15%

### Rationale for Adjustments:
1. **Stock Decrease to 50%**: 
   - This reduction minimizes risk in a high-volatility regime and declining market, preserving capital while reducing exposure to equity volatility.

2. **Bond Increase to 35%**:
   - Increasing bonds further enhances stability and potentially provides income during market uncertainty. Bonds act as a buffer and are more resilient in a declining market.

3. **Cash Increase to 15%**:
   - Raising cash to 15% helps with liquidity and provides dry powder for potential buying opportunities in future market rebounds or allows for flexibility to reallocate swiftly.

### Proposed Rebalancing Trades:
1. **Sell Stocks**: Decrease from 70% to 50%
   - **Action**: Sell 20% of equity.
   - **Result**: This reduces stock exposure and increases stabili

### Display Working Memory

We now display the  agent’s working memory following the execution of the  two previous queries:

In [9]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Stocks**: 70%
- **Bonds**: 20%
- **Cash**: 10%

### Target Allocation: 
Given the high volatility and declining market trend, a more conservative allocation is advisable, prioritizing capital preservation and risk control.

### Recommended Adjusted Allocation:
- **Stocks**: 50%
- **Bonds**: 40%
- **Cash**: 10%

### Rationale:
1. **High Volatility**: The current high volatility regime suggests a need to reduce exposure to riskier assets like stocks.
2. **Declining Market Trend**: A downtrend in the overall market favors a more defensive strategy (increasing bond exposure).

### Proposed Rebalancing Trades:
1. **Sell Stocks**: 

As you can see, all the discussion is stored in the working memory (session).

## User Query 3

Here is another request, still with the session parameter specified during the run:

In [10]:
input_user_3 = """Current Market Situation:
- Volatility Regime: high
- Market Trend: uncertain

What rebalancing action should I take?"""

output_3 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_3,
        session=session_ptf_rebal
    )

print(f"{output_3.final_output}\n")

### Current Portfolio Analysis:
- **Current Portfolio**: 
  - Stocks: 50% 
  - Bonds: 35% 
  - Cash: 15%

### Market Context: 
- **High Volatility**: Indicates a need for risk management.
- **Uncertain Market Trend**: This suggests potential for both upward or downward movement, warranting a cautious approach.

### Recommended Adjusted Allocation:
Given the high volatility and uncertain market context, maintaining a balanced and conservative strategy is advisable.

- **Stocks**: 40%
- **Bonds**: 45%
- **Cash**: 15%

### Rationale:
1. **Stock Reduction to 40%**: 
   - Decreasing stock exposure further protects against volatility, enabling the portfolio to retain growth potential while mitigating risk.
   
2. **Bond Increase to 45%**: 
   - Increasing bonds allows for more stability. Bonds can provide income and a measure of safety in volatile or uncertain markets.

3. **Cash Maintained at 15%**: 
   - Retaining a cash position preserves liquidity, allowing for potential investment oppor

### Display Working Memory

In [11]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Stocks**: 70%
- **Bonds**: 20%
- **Cash**: 10%

### Target Allocation: 
Given the high volatility and declining market trend, a more conservative allocation is advisable, prioritizing capital preservation and risk control.

### Recommended Adjusted Allocation:
- **Stocks**: 50%
- **Bonds**: 40%
- **Cash**: 10%

### Rationale:
1. **High Volatility**: The current high volatility regime suggests a need to reduce exposure to riskier assets like stocks.
2. **Declining Market Trend**: A downtrend in the overall market favors a more defensive strategy (increasing bond exposure).

### Proposed Rebalancing Trades:
1. **Sell Stocks**: 

## User Query 4

In [12]:
input_user_4 = """In this case, I reduce equity to 45%, increase bonds to 40% and keep cash unchanged"""

output_4 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_4,
        session=session_ptf_rebal
    )

print(f"{output_4.final_output}\n")

### Revised Portfolio Analysis:
- **Current Allocation**:
  - Stocks: 50% 
  - Bonds: 35% 
  - Cash: 15%

### Proposed Adjusted Allocation:
- **Stocks**: 45%
- **Bonds**: 40%
- **Cash**: 15%

### Rationale for Adjustments:
1. **Equity Decrease to 45%**:
   - This modest reduction aligns with a strategy to decrease risk while still maintaining some exposure to growth. It allows for potential upside in equity but with reduced risk exposure.

2. **Bond Increase to 40%**:
   - Increasing bond allocation enhances stability and could provide better protection against market fluctuations. This allocation can yield income and contribute to overall risk mitigation.

3. **Cash Maintenance at 15%**:
   - Keeping cash at 15% ensures sufficient liquidity for future opportunities while preserving capital in uncertain market conditions.

### Proposed Rebalancing Trades:
1. **Sell Stocks**: Decrease from 50% to 45%
   - **Action**: Sell 5% of equity.
   - **Impact**: This slight reduction lowers risk 

### Display Working Memory

In [13]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Stocks**: 70%
- **Bonds**: 20%
- **Cash**: 10%

### Target Allocation: 
Given the high volatility and declining market trend, a more conservative allocation is advisable, prioritizing capital preservation and risk control.

### Recommended Adjusted Allocation:
- **Stocks**: 50%
- **Bonds**: 40%
- **Cash**: 10%

### Rationale:
1. **High Volatility**: The current high volatility regime suggests a need to reduce exposure to riskier assets like stocks.
2. **Declining Market Trend**: A downtrend in the overall market favors a more defensive strategy (increasing bond exposure).

### Proposed Rebalancing Trades:
1. **Sell Stocks**: 

## User Query 5

In [14]:
input_user_5 = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

output_5 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_5,
        session=session_ptf_rebal
    )

print(f"{output_5.final_output}\n")

### Current Portfolio Analysis:
- **Current Allocation**:
  - Stocks: 45% 
  - Bonds: 40% 
  - Cash: 15%

### Market Context: 
- **Low Volatility**: Suggests a favorable environment for equities.
- **Bullish Trend**: Indicates potential for continued growth in the stock market.

### Recommended Adjusted Allocation:
Given the low volatility and bullish market conditions, it may be prudent to increase equity exposure while slightly adjusting bond holdings.

- **Stocks**: 55%
- **Bonds**: 35%
- **Cash**: 10%

### Rationale:
1. **Increase Stocks to 55%**:
   - Enhancing equity exposure allows the portfolio to capitalize on growth opportunities during a bullish market. A higher allocation to stocks can increase overall returns in this favorable environment.

2. **Decrease Bonds to 35%**:
   - Reducing bond exposure slightly makes sense since bonds typically offer lower returns in rising market conditions and may have less appeal when stocks are performing well.

3. **Decrease Cash to 10%**:

## User Query 6

In [15]:
input_user_6 = """in this situation, I'll increase stock to 70%, reduce bonds to 25% and cash to 5%"""

output_6 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_6,
        session=session_ptf_rebal
    )

print(f"{output_6.final_output}\n")

### Revised Portfolio Analysis:
- **Current Allocation**:
  - Stocks: 45% 
  - Bonds: 40% 
  - Cash: 15%

### Proposed Adjusted Allocation:
- **Stocks**: 70%
- **Bonds**: 25%
- **Cash**: 5%

### Rationale for Adjustments:
1. **Equity Increase to 70%**:
   - Significantly boosting stock allocation allows for maximum participation in a bullish market, with the potential for higher returns. This aligns well with the prevailing low volatility, which supports more aggressive equity investments.

2. **Bond Decrease to 25%**:
   - Lowering bond exposure reflects the opportunity cost of holding fixed income in a rising market, where equities are expected to outperform.

3. **Cash Decrease to 5%**:
   - Reducing cash reserves to 5% signifies a more aggressive stance, indicating confidence in equity performance. It still maintains a small buffer for liquidity, although it is notably minimal.

### Proposed Rebalancing Trades:
1. **Buy Stocks**: Increase from 45% to 70%
   - **Action**: Buy 25% mo

### Display Final Working Memory

In [16]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Stocks**: 70%
- **Bonds**: 20%
- **Cash**: 10%

### Target Allocation: 
Given the high volatility and declining market trend, a more conservative allocation is advisable, prioritizing capital preservation and risk control.

### Recommended Adjusted Allocation:
- **Stocks**: 50%
- **Bonds**: 40%
- **Cash**: 10%

### Rationale:
1. **High Volatility**: The current high volatility regime suggests a need to reduce exposure to riskier assets like stocks.
2. **Declining Market Trend**: A downtrend in the overall market favors a more defensive strategy (increasing bond exposure).

### Proposed Rebalancing Trades:
1. **Sell Stocks**: 

In [17]:
conversation_history_string = "/n".join([f"{elem[0]}: {elem[1]}" for elem in conversation_history])

conversation_history_string+='user: p'

You can play with the agent yourself:

In [ ]:
# input_user_7 = """ Write here your prompt
# """

# output_7 = await Runner.run(
#         starting_agent=portfolio_agent,
#         input=input_user_7,
#         session=session_ptf_rebal
#     )

# print(f"{output_7.final_output}\n")

Working memory is essential in this context.

Without it, the agent would not retain the initial portfolio allocation, {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}, and would therefore be unable to correctly apply subsequent instructions, such as increasing bonds to 25%, and reducing cash to 5%.

Without access to the initial state, the agent would not compute or recommend an appropriate rebalancing strategy.

# Generate Episodic Memory

I'll show here two methods to generate the episodic memory:

* Using LangChain API
* Using Responses API from OpenAI

## Prompt for Episodic Memory

 We design a detailed prompt that provides clear instructions on the information to extract. We also include examples using a few-shot approach, illustrating what constitutes good and poor summaries, as well as effective and ineffective outcomes.


In [18]:
episodic_memory_prompt_template = """You are an episodic memory agent supporting a principal portfolio-rebalancing agent.
Your job is to create a concise episodic memory of the session:
* Summarize the conversation (in conversation_summary),
* Store a structured trajectory of portfolio allocations and their corresponding market situations, capturing how allocations evolved over time and under which conditions (in portfolio_trajectory).
* Identify what worked (in what_worked) and what didn’t (in what_didnt_work),
* Extract key points (in key_points).

You receive the full conversation history between the user and the principal agent.
The portfolio includes equities, bonds, and cash, and the discussion covers the market situation, portfolio state and outcome, and rebalancing actions.

OUPUT FORMAT:
Return only valid JSON in this exact schema:
{{

	"conversation_summary": string, // 3–5 sentences summarizing: market conditions, portfolio situation, actions taken or discussed.
	"portfolio_allocation_trajectory": string, // Portfolio trajectory recording how portfolio allocations evolve across sessions together with the associated market situations.
	"what_worked": string, //Short analysis of what was effective: alignment with goals, good risk management, coherent rebalancing reasoning, improved diversification, clear decisions.
	"what_didnt_work": string, //Short analysis of frictions: unclear strategy, missed opportunities, over/under-reaction to market signals, concentration risk, unclear constraints, data issues
	"key_points": ["string"] // List of the most important insights or decisions to keep for future sessions.
}}


FEW-SHOT EXAMPLES
Here are some examples of what is good to produce and what is bad to avoid:

GOOD conversation_summary — Example
“Market uncertainty increased due to rising yields and mixed earnings. The portfolio was equity-heavy at 65%, creating higher volatility. The user discussed reducing equity risk and reallocating toward short-duration bonds. A partial rebalance was executed, bringing equities to 55% and increasing cash reserves. The agent emphasized maintaining flexibility given macro uncertainty.”

BAD conversation_summary — Example
“The market was bad. The portfolio changed. Some trades were done. It was about rebalancing.”
(Too vague, no numbers, no context, no actions.)

GOOD what_worked — Example
“The shift from long-duration bonds to shorter maturities improved interest-rate resilience. The reduction of tech overweight aligned well with the user’s lower risk tolerance. The conversation clearly linked macro signals to rebalancing decisions.”

BAD what_worked — Example
“Everything worked well.” (Empty, no link to portfolio or market.)
“The agent was nice.” (Irrelevant.)

GOOD what_didnt_work — Example:
“The user’s liquidity needs were mentioned but not fully integrated into the new allocation. Equity risk remains high despite concerns about earnings volatility. Market data gaps created hesitation about bond exposure.”

BAD what_didnt_work — Example:
“Nothing went wrong.” (Unrealistic; lacks analysis.)
“The user disagreed with the agent.” (Not informative, no portfolio implications.)

General Rules:
- Be concise and factual.
- Do not invent trades or market context not present in the conversation.
- Capture insights, not generic statements.
- Output only valid JSON and nothing else.

Here is the conversation between the user and the principal portfolio-rebalancing agent:
{conversation}
"""

## Working Memory Recap: Conversation History

In [19]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Stocks**: 70%
- **Bonds**: 20%
- **Cash**: 10%

### Target Allocation: 
Given the high volatility and declining market trend, a more conservative allocation is advisable, prioritizing capital preservation and risk control.

### Recommended Adjusted Allocation:
- **Stocks**: 50%
- **Bonds**: 40%
- **Cash**: 10%

### Rationale:
1. **High Volatility**: The current high volatility regime suggests a need to reduce exposure to riskier assets like stocks.
2. **Declining Market Trend**: A downtrend in the overall market favors a more defensive strategy (increasing bond exposure).

### Proposed Rebalancing Trades:
1. **Sell Stocks**: 

### Concat conversation history

In [20]:
conversation_history_string = "\n\n".join([f'{elem[0]}: """{elem[1]}"""' for elem in conversation_history])

print(conversation_history_string)

user: """Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?"""

assistant: """### Current Portfolio Analysis:
- **Stocks**: 70%
- **Bonds**: 20%
- **Cash**: 10%

### Target Allocation: 
Given the high volatility and declining market trend, a more conservative allocation is advisable, prioritizing capital preservation and risk control.

### Recommended Adjusted Allocation:
- **Stocks**: 50%
- **Bonds**: 40%
- **Cash**: 10%

### Rationale:
1. **High Volatility**: The current high volatility regime suggests a need to reduce exposure to riskier assets like stocks.
2. **Declining Market Trend**: A downtrend in the overall market favors a more defensive strategy (increasing bond exposure).

### Proposed Rebalancing Trades:
1. **Sell Stocks**: Decrease exposure from 70% to 50%
   - **Action**: Sell 20% of equity.
   - **Rationale**: Reducing s

## Method 1: LangChain

Install packages

In [21]:
!pip install langchain_openai -q

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [23]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0.7, model="gpt-4o-mini")

### LangChain prompt template

In [24]:
episodic_memory_prompt = ChatPromptTemplate.from_template(episodic_memory_prompt_template)

generate_episodic_memory = episodic_memory_prompt | llm | JsonOutputParser() # Chaining prompt, llm and structured output ==> to be used to generate the episodic memory

In [25]:
print(episodic_memory_prompt.messages[0].prompt.template)

You are an episodic memory agent supporting a principal portfolio-rebalancing agent.
Your job is to create a concise episodic memory of the session:
* Summarize the conversation (in conversation_summary),
* Store a structured trajectory of portfolio allocations and their corresponding market situations, capturing how allocations evolved over time and under which conditions (in portfolio_trajectory).
* Identify what worked (in what_worked) and what didn’t (in what_didnt_work),
* Extract key points (in key_points).

You receive the full conversation history between the user and the principal agent.
The portfolio includes equities, bonds, and cash, and the discussion covers the market situation, portfolio state and outcome, and rebalancing actions.

OUPUT FORMAT:
Return only valid JSON in this exact schema:
{{

	"conversation_summary": string, // 3–5 sentences summarizing: market conditions, portfolio situation, actions taken or discussed.
	"portfolio_allocation_trajectory": string, // Po

### Generate Episodic Memory with LangChain

In [26]:
episodic_memory = generate_episodic_memory.invoke({"conversation": conversation_history_string})
episodic_memory

{'conversation_summary': 'The market was characterized by low volatility and a bullish trend. The user adjusted their portfolio by significantly increasing equity exposure to 70%, reducing bonds to 25%, and decreasing cash to 5%. This aggressive reallocation aims to capitalize on the favorable market conditions, emphasizing growth potential while maintaining minimal liquidity.',
 'portfolio_allocation_trajectory': 'Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15%; Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 35%, Cash 10% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5%.',
 'what_worked': 'The user’s rebalancing strategy effectively aligned with the bullish market conditions, maximizing equity exposure to capitalize on growth while minimizing bond holdings that provide lower returns in such environments.',
 'what_didnt_work

In [30]:
episodic_memory.type()

AttributeError: 'dict' object has no attribute 'type'

The result follows the structured format specified in the prompt. A summary of the conversation is then generated, along with additional analysis highlighting what worked during execution, what did not, and the key elements of the exchange between the agent and the user.


In [ ]:
# You can also add the whole conversation history to the episodic memory.
# Or add only the last 4 messages (user & assistant)
# episodic_memory
# str_json = f"""{'conversation_history': {conversation_history_string},
#   'conversation_summary': 'The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.',
#  'portfolio_allocation_trajectory': 'Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market)',
#  'what_worked': 'The decision to significantly increase stock allocation aligns well with the bullish market trend, leveraging potential for higher returns. The strategy demonstrates effective risk management by reducing bonds and cash in favor of equities, optimizing the portfolio for growth.',
#  'what_didnt_work': 'The substantial reduction in bonds and cash may expose the portfolio to greater risk, particularly if market conditions fluctuate unexpectedly. A lower cash reserve could limit liquidity for future opportunities or risk mitigation.',
#  'key_points': ['Leverage bullish market conditions by increasing equity exposure.',
#   'Maintain a balance between risk and potential returns when adjusting allocations.',
#   'Monitor market conditions closely after significant rebalancing actions.']}"""

In [27]:
# if episodic_memory is a string
import json

if isinstance(episodic_memory, dict):
    episodic_memory_json = episodic_memory

elif isinstance(episodic_memory, str):
    episodic_memory_json = json.loads(episodic_memory)

else:
    raise TypeError(
        f"Tipo no esperado: {type(episodic_memory)}"
    )
# str_json = """{'conversation_summary': 'The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.',
#  'portfolio_allocation_trajectory': 'Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market)',
#  'what_worked': 'The decision to significantly increase stock allocation aligns well with the bullish market trend, leveraging potential for higher returns. The strategy demonstrates effective risk management by reducing bonds and cash in favor of equities, optimizing the portfolio for growth.',
#  'what_didnt_work': 'The substantial reduction in bonds and cash may expose the portfolio to greater risk, particularly if market conditions fluctuate unexpectedly. A lower cash reserve could limit liquidity for future opportunities or risk mitigation.',
#  'key_points': ['Leverage bullish market conditions by increasing equity exposure.',
#   'Maintain a balance between risk and potential returns when adjusting allocations.',
#   'Monitor market conditions closely after significant rebalancing actions.']}"""

# import json
# str_json_fixed = str_json.replace("'", '"')
# episodic_memory_json = json.loads(str_json_fixed)

## Method 2: Responses API from OpenAI

In [28]:
user_prompt = episodic_memory_prompt_template.format(
    conversation=conversation_history_string
)
print(user_prompt)

You are an episodic memory agent supporting a principal portfolio-rebalancing agent.
Your job is to create a concise episodic memory of the session:
* Summarize the conversation (in conversation_summary),
* Store a structured trajectory of portfolio allocations and their corresponding market situations, capturing how allocations evolved over time and under which conditions (in portfolio_trajectory).
* Identify what worked (in what_worked) and what didn’t (in what_didnt_work),
* Extract key points (in key_points).

You receive the full conversation history between the user and the principal agent.
The portfolio includes equities, bonds, and cash, and the discussion covers the market situation, portfolio state and outcome, and rebalancing actions.

OUPUT FORMAT:
Return only valid JSON in this exact schema:
{

	"conversation_summary": string, // 3–5 sentences summarizing: market conditions, portfolio situation, actions taken or discussed.
	"portfolio_allocation_trajectory": string, // Por

In [29]:
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1-mini",
  input=[{"role":"user","content":user_prompt}],
  text = {"format":{"type": "json_object"}},
)

print(json.dumps(json.loads(response.output[0].content[0].text), indent=2))

{
  "conversation_summary": "The portfolio began with a high equity allocation (70%) amid a high volatility and declining market environment. The user progressively reduced stock exposure to more conservative levels (down to 45%) while increasing bonds and cash to enhance stability and liquidity during uncertain or volatile regimes. When market conditions shifted to low volatility with a bullish trend, the user adopted an aggressive stance, increasing equities back up to 70%, reducing bonds to 25%, and lowering cash to 5%, reflecting confidence in growth potential. Rebalancing decisions consistently aligned portfolio risk exposures with evolving market signals, aiming for capital preservation in tough markets and growth capture in favorable ones.",
  "portfolio_allocation_trajectory": "Session 1: Market high volatility, declining trend; Portfolio: Stocks 70%, Bonds 20%, Cash 10%; User reduced stocks to 50%, increased bonds to 35%, cash to 15%. Session 2: Market high volatility, uncerta

In [30]:
episodic_memory = json.dumps(json.loads(response.output[0].content[0].text), indent=2)

# Storing the Episodic Memory:

## ChromaDB client

The generated content, "episodic_memory", is embedded using the text-embedding-3-small model from OpenAI and stored in a vector database, ChromaDB, for efficient similarity-based retrieval.


In [31]:
!pip show \
  chromadb \
  google-ai-generativelanguage \
  googleapis-common-protos \
  grpcio-status \
  protobuf

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 
---
Name: google-ai-generativelanguage
Version: 0.6.15
Summary: Google Ai Generativelanguage API client library
Home-page: https://github.com/googleapis/google-cloud-python/tree/main/packages/google-ai-generativelanguage
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: google-api-core, google-auth, proto-plus

In [32]:
!pip check

ipython 7.34.0 requires jedi, which is not installed.
google-adk 2.7.1 has requirement opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0.
google-adk 2.7.1 has requirement opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0.


In [33]:
!pip install -U \
    google-ai-generativelanguage \
    google-generativeai \
    googleapis-common-protos \
    grpcio-status \
    chromadb

  Using cached google_ai_generativelanguage-0.12.0-py3-none-any.whl.metadata (9.9 kB)
  Using cached googleapis_common_protos-1.75.3-py3-none-any.whl.metadata (8.5 kB)
  Using cached grpcio_status-1.83.1-py3-none-any.whl.metadata (1.2 kB)
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
  Using cached googleapis_common_protos-1.75.2-py3-none-any.whl.metadata (8.5 kB)
  Using cached googleapis_common_protos-1.75.1-py3-none-any.whl.metadata (8.5 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.83.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached grpcio_status-1.82.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached grpcio_status-1.82.1-py3-none-any.whl.metadata (1.2 kB)
  Using cached grpcio_status-1.81.1-py3-none-any.whl.metadata (1.2 kB)
  

In [34]:
!pip install chromadb -q

In [35]:
import chromadb
print(chromadb.__version__)

1.5.9


In [36]:
import chromadb

from chromadb.utils import embedding_functions

# Define an OpenAI embedding function using the specified embedding model
embedding_function  = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY, #Your OpenAI API Key
    model_name="text-embedding-3-small",
)

# Name of the collection used to store episodic memory embeddings
collection_name="episodic_memory_ptf_rebal"

# Create a persistent ChromaDB client with on-disk storage: will be stored in the folder "chroma_ptf_rebal"
persistent_client = chromadb.PersistentClient(path="./chroma_ptf_rebal")

# Create or load the collection and associate it with the embedding function
collection = persistent_client.get_or_create_collection(collection_name, embedding_function=embedding_function )

The collection serves as a container within the vector database where embeddings will be stored.

In [37]:
#Get the different parameters under the ChromaDB client:
persistent_client.get_collection(name='episodic_memory_ptf_rebal').get_model()

Collection(id=UUID('0428fb2f-2c78-410b-8730-43aeb37a0209'), name='episodic_memory_ptf_rebal', configuration_json={'hnsw': {'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': {'type': 'known', 'name': 'openai', 'config': {'api_base': None, 'api_key_env_var': 'OPENAI_API_KEY', 'api_type': None, 'api_version': None, 'default_headers': None, 'deployment_id': None, 'dimensions': None, 'model_name': 'text-embedding-3-small', 'organization_id': None}}}, serialized_schema={'defaults': {'string': {'fts_index': {'enabled': False, 'config': {}}, 'string_inverted_index': {'enabled': True, 'config': {}}}, 'float_list': {'vector_index': {'enabled': False, 'config': {'space': 'cosine', 'embedding_function': {'type': 'known', 'name': 'openai', 'config': {'api_base': None, 'api_key_env_var': 'OPENAI_API_KEY', 'api_type': None, 'api_version': None, 'default_headers': None, 'deployment_id': 

## Add Memory in DB

In [38]:
import json

if isinstance(episodic_memory, str):
    episodic_memory = episodic_memory.strip()

    # Por si viene envuelto en ```json ... ```
    episodic_memory = episodic_memory.removeprefix("```json")
    episodic_memory = episodic_memory.removeprefix("```")
    episodic_memory = episodic_memory.removesuffix("```")
    episodic_memory = episodic_memory.strip()

    episodic_memory = json.loads(episodic_memory)

In [39]:
print(type(episodic_memory))

<class 'dict'>


In [40]:
episodic_memory

{'conversation_summary': 'The portfolio began with a high equity allocation (70%) amid a high volatility and declining market environment. The user progressively reduced stock exposure to more conservative levels (down to 45%) while increasing bonds and cash to enhance stability and liquidity during uncertain or volatile regimes. When market conditions shifted to low volatility with a bullish trend, the user adopted an aggressive stance, increasing equities back up to 70%, reducing bonds to 25%, and lowering cash to 5%, reflecting confidence in growth potential. Rebalancing decisions consistently aligned portfolio risk exposures with evolving market signals, aiming for capital preservation in tough markets and growth capture in favorable ones.',
 'portfolio_allocation_trajectory': 'Session 1: Market high volatility, declining trend; Portfolio: Stocks 70%, Bonds 20%, Cash 10%; User reduced stocks to 50%, increased bonds to 35%, cash to 15%. Session 2: Market high volatility, uncertain t

In [41]:
document_text = f"""
Conversation Summary: {episodic_memory['conversation_summary']}
Portfolio Allocation Trajectory: {episodic_memory['portfolio_allocation_trajectory']}
What Worked: {episodic_memory['what_worked']}
What Didn't Work: {episodic_memory['what_didnt_work']}
Key Points: {', '.join(episodic_memory['key_points'])}
"""

metadata_clean = {
    "conversation_summary": episodic_memory["conversation_summary"],
    "portfolio_allocation_trajectory": episodic_memory["portfolio_allocation_trajectory"],
    "what_worked": episodic_memory["what_worked"],
    "what_didnt_work": episodic_memory["what_didnt_work"],
    "key_points": " | ".join(episodic_memory["key_points"]), #Convert a list to a string to be added in chromDB
}

collection.add(
    ids=["memory_1"],  # unique id
    documents=[document_text],  # main text
    metadatas=[metadata_clean],  # full dict as metadata (JSON-serializable)
)


The generated content, `episodic_memeory`, is stored in two complementary ways:
* as formatted text in the documents field
* as structured JSON in the metadatas field.

Together, these two representations capture the full content of the episodic memory while enabling both semantic search (via embeddings) and structured filtering (via metadata).


## Query Memory

Now, consider starting a new session with the agent and submitting a new query.


As a first step, this query is compared against the episodic memory stored in the vector database using similarity-based distance search:


In [42]:
input_user_init = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

episodic_memory_recap = collection.query(
    query_texts=[input_user_init],
    n_results=3,
)
episodic_memory_recap

{'ids': [['memory_1']],
 'embeddings': None,
 'documents': [["\nConversation Summary: The portfolio began with a high equity allocation (70%) amid a high volatility and declining market environment. The user progressively reduced stock exposure to more conservative levels (down to 45%) while increasing bonds and cash to enhance stability and liquidity during uncertain or volatile regimes. When market conditions shifted to low volatility with a bullish trend, the user adopted an aggressive stance, increasing equities back up to 70%, reducing bonds to 25%, and lowering cash to 5%, reflecting confidence in growth potential. Rebalancing decisions consistently aligned portfolio risk exposures with evolving market signals, aiming for capital preservation in tough markets and growth capture in favorable ones.\nPortfolio Allocation Trajectory: Session 1: Market high volatility, declining trend; Portfolio: Stocks 70%, Bonds 20%, Cash 10%; User reduced stocks to 50%, increased bonds to 35%, cash

We query the vector database to retrieve the three most relevant chunks for the user’s query.

Since we have stored only one episodic memory, we retrieve it and incorporate it into the prompt to help the agent answer the user's question.

In [45]:
# context_memory =
episodic_memory_recap['documents'][0]

["\nConversation Summary: The portfolio began with a high equity allocation (70%) amid a high volatility and declining market environment. The user progressively reduced stock exposure to more conservative levels (down to 45%) while increasing bonds and cash to enhance stability and liquidity during uncertain or volatile regimes. When market conditions shifted to low volatility with a bullish trend, the user adopted an aggressive stance, increasing equities back up to 70%, reducing bonds to 25%, and lowering cash to 5%, reflecting confidence in growth potential. Rebalancing decisions consistently aligned portfolio risk exposures with evolving market signals, aiming for capital preservation in tough markets and growth capture in favorable ones.\nPortfolio Allocation Trajectory: Session 1: Market high volatility, declining trend; Portfolio: Stocks 70%, Bonds 20%, Cash 10%; User reduced stocks to 50%, increased bonds to 35%, cash to 15%. Session 2: Market high volatility, uncertain trend;

## Call the agent

As we are starting new session conversation, let's create a new working memory:

In [46]:
#Create a new session that will persist across runs
session_id = "conversation_ptf_2"
session_ptf_rebal_2 = SQLiteSession(session_id)

In [47]:
input_user_init = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

input_user_with_memory = input_user_init + f""""

Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:
{episodic_memory_recap['documents'][0][0]}

Argue your proposition using past rebalancing episodes if needed.
"""

output_using_memory = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_with_memory,
        session=session_ptf_rebal_2
    )

print(f"{output_using_memory.final_output}\n")

Given the current market situation of low volatility and a bullish trend, here’s a recommended rebalancing action for the portfolio:

### Current Portfolio Allocation
- **Stocks:** 70%
- **Bonds:** 25%
- **Cash:** 5%

### Target Allocation Proposal
Given the continued bullish trend and low volatility, the allocation can further enhance equity exposure but should also ensure risk control and liquidity management, especially if conditions change unexpectedly. A proposed target allocation could be:

- **Stocks:** 75%
- **Bonds:** 20%
- **Cash:** 5%

### Proposed Rebalancing Actions
1. **Increase Equity Exposure:**
   - **Action:** Increase stocks from 70% to 75%.
   - **Change:** +5%.
   - **Rationale:** The market trend is bullish, and increasing equity allocation captures additional upside potential. The previous episodes demonstrated responsiveness to market trends; therefore, this aligns with your bias towards growth in favorable environments.

2. **Minor Reduction in Bonds:**
   - **

As you can see, by incoporating the episodic memory, the agent was able to retrieve the last portfolio allocation (from the first interactions with the agent - working memory Part 1): 70% stocks, 25% bonds, 5% cash. Also, the agent was able to retrieve lessons from past episodes and incorporate them in the current answer.

## Display the new working memory

Here we are displaying the new working memory (session_ptf_rebal_2):

In [48]:
all_items = await session_ptf_rebal_2.get_items()

conversation_history_2 = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history_2.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history_2.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"

Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:

Conversation Summary: The portfolio began with a high equity allocation (70%) amid a high volatility and declining market environment. The user progressively reduced stock exposure to more conservative levels (down to 45%) while increasing bonds and cash to enhance stability and liquidity during uncertain or volatile regimes. When market conditions shifted to low volatility with a bullish trend, the user adopted an aggressive stance, increasing equities back up to 70%, reducing bonds to 25%, and lowering cash to 5%, reflecting confidence in growth potential. Rebalancing decisions consistently aligned portfolio risk exposures with evolving market signals, aiming for capital preservation in tough markets and growth capture in favorable ones.
Portfolio Allocation T

## All in

In this step, we wrap the entire code into functions to improve readability.

* We instantiate a new working session.
* For each input query, we retrieve the most relevant chunks from the episodic memory
* We incorporate these chunks into the prompt, alongside the input query
* The agent answers based on this information
* When we end the discussion with agent, the current working session will be resumed, building the episodic memory, and stored in the vector database, to be used in future interactions.

In [49]:
def concat_working_memory(all_items):
  conversation_history = []
  for i, msg in enumerate(all_items, 1):
      role = msg.get("role", "unknown")
      content = msg.get("content", "")
      if role == "assistant" :
        conversation_history.append((role, content[0]['text']))
      if role == "user":
        conversation_history.append((role, content))
  return conversation_history

In [50]:
def get_episodic_memory(conversation_history):
  conversation_history_string = "\n\n".join([f'{elem[0]}: """{elem[1]}"""' for elem in conversation_history])
  episodic_memory = generate_episodic_memory.invoke({"conversation": conversation_history_string})
  return episodic_memory

In [51]:
def add_data_to_episodic_memory(episodic_memory, collection_id):
  document_text = f"""Conversation Summary: {episodic_memory['conversation_summary']}
  Portfolio Allocation Trajectory: {episodic_memory['portfolio_allocation_trajectory']}
  What Worked: {episodic_memory['what_worked']}
  What Didn't Work: {episodic_memory['what_didnt_work']}
  Key Points: {', '.join(episodic_memory['key_points'])}
  """

  metadata_clean = {
      "conversation_summary": episodic_memory["conversation_summary"],
      "portfolio_allocation_trajectory": episodic_memory["portfolio_allocation_trajectory"],
      "what_worked": episodic_memory["what_worked"],
      "what_didnt_work": episodic_memory["what_didnt_work"],
      "key_points": " | ".join(episodic_memory["key_points"]), #Convert a list to a string to be added in chromDB
  }

  collection.add(
      ids=[collection_id],  # unique id
      documents=[document_text],  # main text
      metadatas=[metadata_clean],  # full dict as metadata (JSON-serializable)
  )

In [52]:
def get_similar_episodes(input_user_init):
  episodic_memory_recap = collection.query(
    query_texts=[input_user_init],
    n_results=3,
  )

  input_user_with_memory = input_user_init + f"""
    Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:
    {episodic_memory_recap['documents'][0][0]}

    Argue your propsition using past rebalacing episodes if needed.
    """
  return input_user_with_memory

In [ ]:
#Initiate Working Memory: This is another working session
session_id = "conversation_ptf_3"
session_ptf_rebal_3 = SQLiteSession(session_id)

#Define the id collection. Conversation will be stored in the episodic memory when the session ends
collection_id = 'memory_2'
all_items = []

while True:
    # Get User's Input
    input_user_init = input("\nUser: ")

    if input_user_init == "exit":
        if all_items!=[]:
          print("End of the discussion, reformulating working memory to be added to episodic memory")
          conversation_history = concat_working_memory(all_items)
          episodic_memory = get_episodic_memory(conversation_history)
          print("---"*50)
          print("Episodic Memory to be stored: \n")
          print(episodic_memory)
          print("---"*50)
          add_data_to_episodic_memory(episodic_memory, collection_id)
        break

    #Extract episodic memory and build final prompt
    input_user_with_memory = get_similar_episodes(input_user_init)


    output_using_memory = await Runner.run(
            starting_agent=portfolio_agent,
            input=input_user_with_memory,
            session=session_ptf_rebal_3
        )

    print(f"{output_using_memory.final_output}\n")

    all_items = await session_ptf_rebal_3.get_items()


# Here are the 2 queries inputed during this run:

# Query #1
# input_user_init = """Current Market Situation:
# - Volatility Regime: low
# - Market Trend: bullish
# What rebalancing action should I take?"""

# Query #2
# I will not change stock exposure, however I'll decrease the bonds to 20% and increase the cash to 10%

#At the end of the discussion, the episodic memory will be built, using the current working session, and stored in the vector database.


User: Current Market Situaation: Volatil
Given the current market situation characterized by high volatility, I'll focus on rebalancing the portfolio to mitigate risk while seeking a reasonable opportunity for growth.

### Current Portfolio Background
- **Equities**: 45%
- **Bonds**: 40%
- **Cash**: 15%
  
### Market Context
- **Volatility**: High
- **Trend**: Uncertain

### Proposed Rebalancing Actions
1. **Equities**: **Reduce from 45% to 40%** (-5%)
   - **Rationale**: In past rebalancing episodes, equity exposure was reduced in high volatility conditions. With the current market uncertainty, reducing exposure from 45% to 40% aligns with the goal of risk mitigation while still maintaining some equity exposure to capture potential bullish opportunities.

2. **Bonds**: **Increase from 40% to 45%** (+5%)
   - **Rationale**: Increasing bond allocation to 45% serves to enhance portfolio stability. Historically, an increase in bonds during high-volatility scenarios has effectively reduce

In [ ]:
all_items

In [ ]:
collection.get(ids=[collection_id])

In [ ]:
collection.get(
    ids=[collection_id],
    include=["embeddings"]
)


# Semantic Memory

Using the same logic (then in episodic memory) of appending the current context with new data, your can extract, using **RAG**, new information such as **facts** or **concepts** from a knowledge database.


This semantic memory is not about past experiences, but it's more about external knowledge your agent may need.

In porftolio rebalancing, this could mean adding **financial analysis** about **sectors** or **macro economic conditions**.

This kind of information can be stored in vector databases using RAG, allowing you to retrieve the most relevant insights when needed (related to the user query) and add them to the agent's context.

# Procedural Memory

A procedural memory is more "how to" perform a certain action.

For example, it could be, for a portfolio rebalancing:

* Identify the current portfolio allocation
* Compare it against the target allocation
* Calculate deviations for each asset class or sector
* Determine required buy/sell adjustments
* Check risk constraints (max exposure, volatility limits, liquidity)
* Propose rebalancing trades

